# 🚢 تجارت‌یار — اجرا روی Colab (آپلود مستقیم ZIP)

این دفترچه بدون نیاز به GitHub، بدون `npm install` و بدون build، سامانه را بالا می‌آورد؛ چون خروجی نهایی (`dist`) از قبل داخل فایل ZIP است.

**مراحل — هر سلول را به ترتیب با `Ctrl+Enter` اجرا کنید:**
1. نصب Node.js ۲۰ (سریع)
2. آپلود فایل `Tejaratyarr.zip` (وقتی پنجره‌ی انتخاب فایل باز شد، فایل را انتخاب کنید)
3. استخراج و آماده‌سازی
4. اجرای سرور
5. بررسی سلامت
6. 👀 **نمایش فوری در همین Colab** (بدون تونل)
7. دانلود cloudflared (فقط اگر لینک قابل ارسال می‌خواهید)
8. راه‌اندازی تونل (بلافاصله تمام می‌شود)
9. دریافت لینک عمومی

> برای **دیدن سریع برنامه** فقط تا سلول ۶ کافی است. برای **لینک قابل ارسال به استاد** سلول‌های ۷ تا ۹ را اجرا کنید.

In [ ]:
%%bash
node -v 2>/dev/null | grep -q "^v1[89]" && echo "node already present: $(node -v)" || (curl -fsSL https://nodejs.org/dist/v20.18.1/node-v20.18.1-linux-x64.tar.xz -o /tmp/node.txz && tar -xJf /tmp/node.txz -C /usr/local --strip-components=1 && echo "node installed: $(node -v)")

In [ ]:
from google.colab import files

print("لطفاً فایل Tejaratyarr.zip را انتخاب و آپلود کنید:")
uploaded = files.upload()
print("آپلود شد:", list(uploaded.keys()))


In [ ]:
%%bash
Z=$(ls -t /content/*.zip 2>/dev/null | head -1); echo "zip file: $Z"; rm -rf /content/Tejaratyarr; unzip -o "$Z" -d /content; echo "---"; ls -la /content/Tejaratyarr/dist/server.cjs

In [ ]:
%%bash
cd /content/Tejaratyarr && NODE_ENV=production setsid nohup node dist/server.cjs > /content/server.log 2>&1 < /dev/null &

In [ ]:
%%bash
sleep 4; curl -s http://localhost:3000/api/health; echo

In [ ]:
from google.colab import output

# نمایش برنامه داخل همین دفترچه (بدون تونل و بدون انتظار)
output.serve_kernel_port_as_window(3000)


In [ ]:
%%bash
cd /content/Tejaratyarr && (test -x cloudflared || curl -L --progress-bar -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64) && chmod +x cloudflared && ./cloudflared --version

In [ ]:
import subprocess

subprocess.run("pkill -f 'cloudflared tunnel' || true", shell=True)

p = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:3000",
     "--no-autoupdate", "--logfile", "/content/cloudflared.log"],
    cwd="/content/Tejaratyarr",
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    stdin=subprocess.DEVNULL,
    start_new_session=True,
)
print("cloudflared started (PID", p.pid, ") — next cell prints the link.")


In [ ]:
import time, re

print("waiting for the tunnel URL (usually 10-30 seconds)...")
url = None
for _ in range(45):
    try:
        with open("/content/cloudflared.log", "r", errors="ignore") as f:
            log = f.read()
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
        if m:
            url = m.group(0)
            break
    except FileNotFoundError:
        pass
    time.sleep(2)

if url:
    print()
    print("LINK:", url)
else:
    print()
    print("link not ready yet — last lines of cloudflared log:")
    try:
        with open("/content/cloudflared.log", "r", errors="ignore") as f:
            print(f.read()[-2000:])
    except Exception as e:
        print(e)


## 📌 نکات

- **نمایش فوری:** سلول ۶ برنامه را داخل خود Colab نشان می‌دهد (برای ارائه‌ی زنده روی صفحه‌ی خودتان کافی است).
- **لینک عمومی:** سلول ۹ یک لینک `trycloudflare` می‌دهد که تا وقتی Colab روشن است قابل ارسال به دیگران است.
- داده‌ها در Colab موقتی است؛ برای استفاده‌ی واقعی روی سرور خودتان اجرا کنید.
- فعال‌سازی هوش مصنوعی Gemini: قبل از سلول ۴، `GEMINI_API_KEY` را تنظیم کنید.
- اجرای محلی: `npm install` سپس `npm run dev` (پورت ۳۰۰۰).
- اجرای تولید محلی: `npm run build` سپس `NODE_ENV=production node dist/server.cjs`.

## 🛠 رفع اشکال
- **آپلود نشد؟** سلول ۲ را دوباره اجرا کنید و مطمئن شوید فایل با نام zip انتخاب می‌شود.
- **سرور بالا نیامد؟** سلول ۵ را اجرا کنید؛ اگر خطا بود سلول ۴ را دوباره اجرا کنید.
- **لینک چاپ نشد؟** سلول ۸ و سپس ۹ را دوباره اجرا کنید.